# Analysis - Interactive Version

This notebook is an interactive version of `analysis.py` that breaks down the analysis into inspectable cells.

## Purpose

Analyzes factor computation results to produce:
- LaTeX tables for Sharpe ratios and Hansen-Jagannathan distances
- Boxplot figures comparing different methods
- PDF compilation of all results

## Recipe

For each month in each panel:
1. `sharpe = mean / stdev`
2. `hjd_sq = (sdf_ret - xret)^2`

Then aggregate:
- Sharpe: mean across panels of (mean across months)
- HJD: mean across panels of sqrt(mean across months of hjd_sq)

## Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from typing import Dict, List
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Imports complete!")

In [ ]:
# Add parent directory to path for imports
parent_dir = Path(os.getcwd())
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from config import DATA_DIR

# Output directories
SCRIPT_DIR = parent_dir
TABLES_DIR = SCRIPT_DIR / "tables"
FIGURES_DIR = SCRIPT_DIR / "figures"

# Create output directories
TABLES_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Tables will be saved to: {TABLES_DIR}")
print(f"Figures will be saved to: {FIGURES_DIR}")

## Configuration

In [ ]:
# Models to analyze
MODELS = ['bgn', 'kp14', 'gs21']

# Which model to analyze (change this to switch models)
MODEL = 'bgn'  # Options: 'bgn', 'kp14', 'gs21'

print(f"Analyzing model: {MODEL}")

## 1. Discover Available Panels

In [ ]:
# Scan for available panels by looking at fama files
pattern = os.path.join(DATA_DIR, f"{MODEL}_*_fama.pkl")
fama_files = glob.glob(pattern)

panels = set()
for filepath in fama_files:
    filename = Path(filepath).stem
    parts = filename.split('_')
    if len(parts) >= 3 and parts[1].isdigit():
        panels.add(int(parts[1]))

panel_indices = sorted(list(panels))

print(f"Found {len(panel_indices)} panels for {MODEL.upper()}: {panel_indices}")

if len(panel_indices) == 0:
    print("\nNo panels found! Make sure to run:")
    print(f"  python run_fama.py {MODEL}_0")
    print(f"  python run_dkkm.py {MODEL}_0 360")
    print(f"  python run_ipca.py {MODEL}_0 1")

## 2. Load and Process Fama Results

In [ ]:
# Load Fama results for all panels
fama_all_data = []

for panel_idx in panel_indices:
    panel_id = f"{MODEL}_{panel_idx}"
    fama_file = os.path.join(DATA_DIR, f"{panel_id}_fama.pkl")
    
    if not os.path.exists(fama_file):
        print(f"  Skipping {panel_id}: file not found")
        continue
    
    # Load file
    with open(fama_file, 'rb') as f:
        fama_data = pickle.load(f)
    
    fama_stats = fama_data.get('fama_stats')
    if fama_stats is None or len(fama_stats) == 0:
        print(f"  Skipping {panel_id}: no stats found")
        continue
    
    print(f"  Loaded {panel_id}: {len(fama_stats)} observations")
    
    # Compute sharpe and hjd_sq for each month
    fama_stats = fama_stats.copy()
    fama_stats['sharpe'] = fama_stats['mean'] / fama_stats['stdev']
    fama_stats['hjd_sq'] = (fama_stats['sdf_ret'] - fama_stats['xret'])**2
    fama_stats['panel'] = panel_idx
    
    fama_all_data.append(fama_stats[['panel', 'month', 'method', 'alpha', 'sharpe', 'hjd_sq']])

if fama_all_data:
    fama_df = pd.concat(fama_all_data, ignore_index=True)
    print(f"\nTotal Fama observations: {len(fama_df)}")
    print(f"Panels: {sorted(fama_df['panel'].unique())}")
    print(f"Methods: {sorted(fama_df['method'].unique())}")
    print(f"Alphas: {sorted(fama_df['alpha'].unique())}")
else:
    fama_df = pd.DataFrame()
    print("\nNo Fama data found!")

In [ ]:
# Inspect Fama data
if len(fama_df) > 0:
    print("Sample of Fama data:")
    display(fama_df.head(20))
    
    print("\nSummary statistics:")
    display(fama_df.groupby(['method', 'alpha'])[['sharpe', 'hjd_sq']].describe())

## 3. Create Fama Table

In [ ]:
# Create Fama table: FFC and FMR sharpe and hjd
if len(fama_df) == 0:
    print(f"No Fama data for {MODEL}")
else:
    alphas = sorted(fama_df['alpha'].unique())
    print(f"Alphas: {alphas}")
    
    # Create table data
    table_data = []
    for alpha in alphas:
        alpha_data = fama_df[fama_df['alpha'] == alpha]
        row = {'alpha': alpha}
        
        # FFC sharpe: mean across panels of (mean across months)
        ffc_data = alpha_data[alpha_data['method'] == 'ff']
        if len(ffc_data) > 0:
            ffc_panel_sharpe = ffc_data.groupby('panel')['sharpe'].mean()
            row['FFC_sharpe'] = ffc_panel_sharpe.mean()
            print(f"  Alpha {alpha:.1e} FFC: {len(ffc_data)} obs, {len(ffc_panel_sharpe)} panels, sharpe={row['FFC_sharpe']:.4f}")
        else:
            row['FFC_sharpe'] = np.nan
        
        # FMR sharpe: mean across panels of (mean across months)
        fmr_data = alpha_data[alpha_data['method'] == 'fm']
        if len(fmr_data) > 0:
            fmr_panel_sharpe = fmr_data.groupby('panel')['sharpe'].mean()
            row['FMR_sharpe'] = fmr_panel_sharpe.mean()
            print(f"  Alpha {alpha:.1e} FMR: {len(fmr_data)} obs, {len(fmr_panel_sharpe)} panels, sharpe={row['FMR_sharpe']:.4f}")
        else:
            row['FMR_sharpe'] = np.nan
        
        # FFC hjd: mean across panels of sqrt(mean across months of hjd_sq)
        if len(ffc_data) > 0:
            ffc_panel_hjd = ffc_data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
            row['FFC_hjd'] = ffc_panel_hjd.mean()
        else:
            row['FFC_hjd'] = np.nan
        
        # FMR hjd: mean across panels of sqrt(mean across months of hjd_sq)
        if len(fmr_data) > 0:
            fmr_panel_hjd = fmr_data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
            row['FMR_hjd'] = fmr_panel_hjd.mean()
        else:
            row['FMR_hjd'] = np.nan
        
        table_data.append(row)
    
    fama_table = pd.DataFrame(table_data).set_index('alpha')
    print("\nFama Table:")
    display(fama_table)

In [ ]:
# Save Fama table to LaTeX
if len(fama_df) > 0:
    latex = fama_table.to_latex(float_format="%.4f", na_rep="--",
                                 column_format='r' + 'r'*len(fama_table.columns),
                                 escape=False)
    
    output_path = TABLES_DIR / f"{MODEL}_fama.tex"
    with open(output_path, 'w') as f:
        f.write(latex)
    
    print(f"Saved to: {output_path}")

## 4. Load and Process DKKM Results

In [ ]:
# Load DKKM results for all panels
dkkm_all_data = []

for panel_idx in panel_indices:
    panel_id = f"{MODEL}_{panel_idx}"
    pattern = os.path.join(DATA_DIR, f"{panel_id}_dkkm_*.pkl")
    
    for filepath in glob.glob(pattern):
        filename = Path(filepath).stem
        parts = filename.split('_')
        
        if len(parts) < 4:
            continue
        
        try:
            nfeatures = int(parts[3])
        except ValueError:
            continue
        
        # Load file
        with open(filepath, 'rb') as f:
            dkkm_data = pickle.load(f)
        
        dkkm_stats = dkkm_data.get('dkkm_stats')
        if dkkm_stats is None or len(dkkm_stats) == 0:
            continue
        
        print(f"  Loaded {panel_id}_dkkm_{nfeatures}: {len(dkkm_stats)} observations")
        
        # Compute sharpe and hjd_sq
        dkkm_stats = dkkm_stats.copy()
        dkkm_stats['sharpe'] = dkkm_stats['mean'] / dkkm_stats['stdev']
        dkkm_stats['hjd_sq'] = (dkkm_stats['sdf_ret'] - dkkm_stats['xret'])**2
        dkkm_stats['panel'] = panel_idx
        dkkm_stats['num_factors'] = nfeatures
        
        dkkm_all_data.append(dkkm_stats[['panel', 'month', 'alpha', 'num_factors', 'sharpe', 'hjd_sq']])

if dkkm_all_data:
    dkkm_df = pd.concat(dkkm_all_data, ignore_index=True)
    print(f"\nTotal DKKM observations: {len(dkkm_df)}")
    print(f"Panels: {sorted(dkkm_df['panel'].unique())}")
    print(f"Alphas: {sorted(dkkm_df['alpha'].unique())}")
    print(f"Number of features: {sorted(dkkm_df['num_factors'].unique())}")
else:
    dkkm_df = pd.DataFrame()
    print("\nNo DKKM data found!")

In [ ]:
# Inspect DKKM data
if len(dkkm_df) > 0:
    print("Sample of DKKM data:")
    display(dkkm_df.head(20))
    
    print("\nSummary statistics:")
    display(dkkm_df.groupby(['alpha', 'num_factors'])[['sharpe', 'hjd_sq']].describe())

## 5. Create DKKM Tables

In [ ]:
# Create DKKM sharpe table
if len(dkkm_df) == 0:
    print(f"No DKKM data for {MODEL}")
else:
    alphas = sorted(dkkm_df['alpha'].unique())
    num_factors_vals = sorted(dkkm_df['num_factors'].unique())
    
    print(f"Alphas: {alphas}")
    print(f"Number of features: {num_factors_vals}")
    
    # Create sharpe table
    sharpe_table = []
    for alpha in alphas:
        row = {'alpha': alpha}
        alpha_data = dkkm_df[dkkm_df['alpha'] == alpha]
        
        for nf in num_factors_vals:
            nf_data = alpha_data[alpha_data['num_factors'] == nf]
            if len(nf_data) > 0:
                panel_sharpe = nf_data.groupby('panel')['sharpe'].mean()
                row[nf] = panel_sharpe.mean()
                print(f"  Alpha {alpha:.1e}, k={nf}: {len(nf_data)} obs, sharpe={row[nf]:.4f}")
            else:
                row[nf] = np.nan
        
        sharpe_table.append(row)
    
    dkkm_sharpe_df = pd.DataFrame(sharpe_table).set_index('alpha')
    print("\nDKKM Sharpe Table:")
    display(dkkm_sharpe_df)

In [ ]:
# Create DKKM hjd table
if len(dkkm_df) > 0:
    hjd_table = []
    for alpha in alphas:
        row = {'alpha': alpha}
        alpha_data = dkkm_df[dkkm_df['alpha'] == alpha]
        
        for nf in num_factors_vals:
            nf_data = alpha_data[alpha_data['num_factors'] == nf]
            if len(nf_data) > 0:
                panel_hjd = nf_data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
                row[nf] = panel_hjd.mean()
            else:
                row[nf] = np.nan
        
        hjd_table.append(row)
    
    dkkm_hjd_df = pd.DataFrame(hjd_table).set_index('alpha')
    print("DKKM HJD Table:")
    display(dkkm_hjd_df)

In [ ]:
# Save DKKM tables
if len(dkkm_df) > 0:
    # Save sharpe table
    latex = dkkm_sharpe_df.to_latex(float_format="%.4f", na_rep="--",
                                     column_format='r' + 'r'*len(dkkm_sharpe_df.columns),
                                     escape=False)
    output_path = TABLES_DIR / f"{MODEL}_dkkm_sharpe.tex"
    with open(output_path, 'w') as f:
        f.write(latex)
    print(f"Saved sharpe table to: {output_path}")
    
    # Save hjd table
    latex = dkkm_hjd_df.to_latex(float_format="%.4f", na_rep="--",
                                  column_format='r' + 'r'*len(dkkm_hjd_df.columns),
                                  escape=False)
    output_path = TABLES_DIR / f"{MODEL}_dkkm_hjd.tex"
    with open(output_path, 'w') as f:
        f.write(latex)
    print(f"Saved hjd table to: {output_path}")

## 6. Load and Process IPCA Results

In [ ]:
# Load IPCA results for all panels
ipca_all_data = []

for panel_idx in panel_indices:
    panel_id = f"{MODEL}_{panel_idx}"
    pattern = os.path.join(DATA_DIR, f"{panel_id}_ipca_*.pkl")
    
    for filepath in glob.glob(pattern):
        filename = Path(filepath).stem
        parts = filename.split('_')
        
        if len(parts) < 4:
            continue
        
        try:
            K = int(parts[3])
        except ValueError:
            continue
        
        # Load file
        with open(filepath, 'rb') as f:
            ipca_data = pickle.load(f)
        
        ipca_stats = ipca_data.get('ipca_stats')
        if ipca_stats is None or len(ipca_stats) == 0:
            continue
        
        print(f"  Loaded {panel_id}_ipca_{K}: {len(ipca_stats)} observations")
        
        # Compute sharpe and hjd_sq
        ipca_stats = ipca_stats.copy()
        ipca_stats['sharpe'] = ipca_stats['mean'] / ipca_stats['stdev']
        ipca_stats['hjd_sq'] = (ipca_stats['sdf_ret'] - ipca_stats['xret'])**2
        ipca_stats['panel'] = panel_idx
        ipca_stats['num_factors'] = K
        
        ipca_all_data.append(ipca_stats[['panel', 'month', 'alpha', 'num_factors', 'sharpe', 'hjd_sq']])

if ipca_all_data:
    ipca_df = pd.concat(ipca_all_data, ignore_index=True)
    print(f"\nTotal IPCA observations: {len(ipca_df)}")
    print(f"Panels: {sorted(ipca_df['panel'].unique())}")
    print(f"Alphas: {sorted(ipca_df['alpha'].unique())}")
    print(f"Number of factors (K): {sorted(ipca_df['num_factors'].unique())}")
else:
    ipca_df = pd.DataFrame()
    print("\nNo IPCA data found!")

In [ ]:
# Inspect IPCA data
if len(ipca_df) > 0:
    print("Sample of IPCA data:")
    display(ipca_df.head(20))
    
    print("\nSummary statistics:")
    display(ipca_df.groupby(['alpha', 'num_factors'])[['sharpe', 'hjd_sq']].describe())

## 7. Create IPCA Tables

In [ ]:
# Create IPCA sharpe table
if len(ipca_df) == 0:
    print(f"No IPCA data for {MODEL}")
else:
    alphas = sorted(ipca_df['alpha'].unique())
    num_factors_vals = sorted(ipca_df['num_factors'].unique())
    
    print(f"Alphas: {alphas}")
    print(f"Number of factors (K): {num_factors_vals}")
    
    # Create sharpe table
    sharpe_table = []
    for alpha in alphas:
        row = {'alpha': alpha}
        alpha_data = ipca_df[ipca_df['alpha'] == alpha]
        
        for K in num_factors_vals:
            K_data = alpha_data[alpha_data['num_factors'] == K]
            if len(K_data) > 0:
                panel_sharpe = K_data.groupby('panel')['sharpe'].mean()
                row[K] = panel_sharpe.mean()
                print(f"  Alpha {alpha:.1e}, K={K}: {len(K_data)} obs, sharpe={row[K]:.4f}")
            else:
                row[K] = np.nan
        
        sharpe_table.append(row)
    
    ipca_sharpe_df = pd.DataFrame(sharpe_table).set_index('alpha')
    print("\nIPCA Sharpe Table:")
    display(ipca_sharpe_df)

In [ ]:
# Create IPCA hjd table
if len(ipca_df) > 0:
    hjd_table = []
    for alpha in alphas:
        row = {'alpha': alpha}
        alpha_data = ipca_df[ipca_df['alpha'] == alpha]
        
        for K in num_factors_vals:
            K_data = alpha_data[alpha_data['num_factors'] == K]
            if len(K_data) > 0:
                panel_hjd = K_data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
                row[K] = panel_hjd.mean()
            else:
                row[K] = np.nan
        
        hjd_table.append(row)
    
    ipca_hjd_df = pd.DataFrame(hjd_table).set_index('alpha')
    print("IPCA HJD Table:")
    display(ipca_hjd_df)

In [ ]:
# Save IPCA tables
if len(ipca_df) > 0:
    # Save sharpe table
    latex = ipca_sharpe_df.to_latex(float_format="%.4f", na_rep="--",
                                     column_format='r' + 'r'*len(ipca_sharpe_df.columns),
                                     escape=False)
    output_path = TABLES_DIR / f"{MODEL}_ipca_sharpe.tex"
    with open(output_path, 'w') as f:
        f.write(latex)
    print(f"Saved sharpe table to: {output_path}")
    
    # Save hjd table
    latex = ipca_hjd_df.to_latex(float_format="%.4f", na_rep="--",
                                  column_format='r' + 'r'*len(ipca_hjd_df.columns),
                                  escape=False)
    output_path = TABLES_DIR / f"{MODEL}_ipca_hjd.tex"
    with open(output_path, 'w') as f:
        f.write(latex)
    print(f"Saved hjd table to: {output_path}")

## 8. Visualizations - Fama Boxplots

In [ ]:
# Fama Sharpe Ratio Boxplot
if len(fama_df) > 0:
    alphas = sorted(fama_df['alpha'].unique())
    
    boxplot_data = []
    labels = []
    
    for alpha in alphas:
        for method, method_label in [('ff', 'FFC'), ('fm', 'FMR')]:
            data = fama_df[(fama_df['alpha'] == alpha) & (fama_df['method'] == method)]
            if len(data) > 0:
                # Compute Sharpe per panel
                panel_sharpe = data.groupby('panel')['sharpe'].mean()
                boxplot_data.append(panel_sharpe.values)
                labels.append(f"{method_label}\n$\\alpha$={alpha:.1e}")
    
    # Check if all groups have only 1 point
    all_single = all(len(d) == 1 for d in boxplot_data)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if all_single:
        # Plot as points
        positions = range(1, len(boxplot_data) + 1)
        values = [d[0] for d in boxplot_data]
        ax.scatter(positions, values, s=100, alpha=0.7)
    else:
        # Plot as boxplot
        bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
    
    ax.set_xlabel('Method and Alpha', fontsize=12)
    ax.set_ylabel('Sharpe Ratio', fontsize=12)
    ax.set_title(f'{MODEL.upper()} - Fama Sharpe Ratios', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Save
    output_path = FIGURES_DIR / f"{MODEL}_fama_sharpe.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved to: {output_path}")
    
    plt.show()

In [ ]:
# Fama HJD Boxplot
if len(fama_df) > 0:
    boxplot_data = []
    labels = []
    
    for alpha in alphas:
        for method, method_label in [('ff', 'FFC'), ('fm', 'FMR')]:
            data = fama_df[(fama_df['alpha'] == alpha) & (fama_df['method'] == method)]
            if len(data) > 0:
                # Compute HJD per panel
                panel_hjd = data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
                boxplot_data.append(panel_hjd.values)
                labels.append(f"{method_label}\n$\\alpha$={alpha:.1e}")
    
    all_single = all(len(d) == 1 for d in boxplot_data)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if all_single:
        positions = range(1, len(boxplot_data) + 1)
        values = [d[0] for d in boxplot_data]
        ax.scatter(positions, values, s=100, alpha=0.7)
    else:
        bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightcoral')
    
    ax.set_xlabel('Method and Alpha', fontsize=12)
    ax.set_ylabel('Hansen-Jagannathan Distance', fontsize=12)
    ax.set_title(f'{MODEL.upper()} - Fama HJD', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Save
    output_path = FIGURES_DIR / f"{MODEL}_fama_hjd.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved to: {output_path}")
    
    plt.show()

## 9. Visualizations - DKKM Boxplots

In [ ]:
# DKKM Sharpe Ratio Boxplot
if len(dkkm_df) > 0:
    alphas = sorted(dkkm_df['alpha'].unique())
    num_factors_vals = sorted(dkkm_df['num_factors'].unique())
    
    boxplot_data = []
    labels = []
    
    for alpha in alphas:
        for nf in num_factors_vals:
            data = dkkm_df[(dkkm_df['alpha'] == alpha) & (dkkm_df['num_factors'] == nf)]
            if len(data) > 0:
                panel_sharpe = data.groupby('panel')['sharpe'].mean()
                boxplot_data.append(panel_sharpe.values)
                labels.append(f"$\\alpha$={alpha:.1e}\nn={nf}")
    
    all_single = all(len(d) == 1 for d in boxplot_data)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    if all_single:
        positions = range(1, len(boxplot_data) + 1)
        values = [d[0] for d in boxplot_data]
        ax.scatter(positions, values, s=100, alpha=0.7)
    else:
        bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightgreen')
    
    ax.set_xlabel('Alpha and Number of Features', fontsize=12)
    ax.set_ylabel('Sharpe Ratio', fontsize=12)
    ax.set_title(f'{MODEL.upper()} - DKKM Sharpe Ratios', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    output_path = FIGURES_DIR / f"{MODEL}_dkkm_sharpe.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved to: {output_path}")
    
    plt.show()

In [ ]:
# DKKM HJD Boxplot
if len(dkkm_df) > 0:
    boxplot_data = []
    labels = []
    
    for alpha in alphas:
        for nf in num_factors_vals:
            data = dkkm_df[(dkkm_df['alpha'] == alpha) & (dkkm_df['num_factors'] == nf)]
            if len(data) > 0:
                panel_hjd = data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
                boxplot_data.append(panel_hjd.values)
                labels.append(f"$\\alpha$={alpha:.1e}\nn={nf}")
    
    all_single = all(len(d) == 1 for d in boxplot_data)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    if all_single:
        positions = range(1, len(boxplot_data) + 1)
        values = [d[0] for d in boxplot_data]
        ax.scatter(positions, values, s=100, alpha=0.7)
    else:
        bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightyellow')
    
    ax.set_xlabel('Alpha and Number of Features', fontsize=12)
    ax.set_ylabel('Hansen-Jagannathan Distance', fontsize=12)
    ax.set_title(f'{MODEL.upper()} - DKKM HJD', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    output_path = FIGURES_DIR / f"{MODEL}_dkkm_hjd.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved to: {output_path}")
    
    plt.show()

## 10. Visualizations - IPCA Boxplots

In [ ]:
# IPCA Sharpe Ratio Boxplot
if len(ipca_df) > 0:
    alphas = sorted(ipca_df['alpha'].unique())
    num_factors_vals = sorted(ipca_df['num_factors'].unique())
    
    boxplot_data = []
    labels = []
    
    for alpha in alphas:
        for K in num_factors_vals:
            data = ipca_df[(ipca_df['alpha'] == alpha) & (ipca_df['num_factors'] == K)]
            if len(data) > 0:
                panel_sharpe = data.groupby('panel')['sharpe'].mean()
                boxplot_data.append(panel_sharpe.values)
                labels.append(f"$\\alpha$={alpha:.1e}\nK={K}")
    
    all_single = all(len(d) == 1 for d in boxplot_data)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    if all_single:
        positions = range(1, len(boxplot_data) + 1)
        values = [d[0] for d in boxplot_data]
        ax.scatter(positions, values, s=100, alpha=0.7)
    else:
        bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lavender')
    
    ax.set_xlabel('Alpha and Number of Factors', fontsize=12)
    ax.set_ylabel('Sharpe Ratio', fontsize=12)
    ax.set_title(f'{MODEL.upper()} - IPCA Sharpe Ratios', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    output_path = FIGURES_DIR / f"{MODEL}_ipca_sharpe.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved to: {output_path}")
    
    plt.show()

In [ ]:
# IPCA HJD Boxplot
if len(ipca_df) > 0:
    boxplot_data = []
    labels = []
    
    for alpha in alphas:
        for K in num_factors_vals:
            data = ipca_df[(ipca_df['alpha'] == alpha) & (ipca_df['num_factors'] == K)]
            if len(data) > 0:
                panel_hjd = data.groupby('panel')['hjd_sq'].apply(lambda x: np.sqrt(x.mean()))
                boxplot_data.append(panel_hjd.values)
                labels.append(f"$\\alpha$={alpha:.1e}\nK={K}")
    
    all_single = all(len(d) == 1 for d in boxplot_data)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    if all_single:
        positions = range(1, len(boxplot_data) + 1)
        values = [d[0] for d in boxplot_data]
        ax.scatter(positions, values, s=100, alpha=0.7)
    else:
        bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('lightpink')
    
    ax.set_xlabel('Alpha and Number of Factors', fontsize=12)
    ax.set_ylabel('Hansen-Jagannathan Distance', fontsize=12)
    ax.set_title(f'{MODEL.upper()} - IPCA HJD', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    output_path = FIGURES_DIR / f"{MODEL}_ipca_hjd.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved to: {output_path}")
    
    plt.show()

## Summary

In [ ]:
print("="*70)
print(f"Analysis Complete for {MODEL.upper()}")
print("="*70)
print()
print(f"Tables saved to: {TABLES_DIR}/")
print(f"Figures saved to: {FIGURES_DIR}/")
print()
print("Generated files:")
for f in sorted(TABLES_DIR.glob(f"{MODEL}_*.tex")):
    print(f"  {f.name}")
for f in sorted(FIGURES_DIR.glob(f"{MODEL}_*.png")):
    print(f"  {f.name}")